In [3]:
from langchain_community.document_loaders import WikipediaLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

In [4]:
# 1. Load and chunk your dataset
chunk_size = 300
chunk_overlap = 100

# loading data
loader = WikipediaLoader(query="YS Jagan Mohan Reddy", load_max_docs=5)
documents = loader.load()

# text splitting
text_splitter = RecursiveCharacterTextSplitter(chunk_size = chunk_size, chunk_overlap = chunk_overlap)
docs = text_splitter.split_documents(documents=documents)
docs

[Document(metadata={'title': 'Y. S. Jagan Mohan Reddy', 'summary': "Yeduguri Sandinti Jagan Mohan Reddy (born 21 December 1972), also known mononymously as Jagan, is an Indian politician and a Member of Legislative Assembly representing Pulivendula Assembly constituency in the Andhra Pradesh Legislative assembly. He previously served as the 17th Chief Minister of Andhra Pradesh. He is the founding president of YSR Congress Party. He is also the son of Y. S. Rajasekhara Reddy, former Chief Minister of Andhra Pradesh and Y. S. Vijayamma. He is also the brother of APCC president Y. S. Sharmila.\nJagan Mohan Reddy started his political career in the Indian National Congress and was elected as the Member of Parliament of Kadapa in 2009.  After his father's death due to a helicopter crash in 2009, he started an Odarpu Yatra (a consoling tour) across the state. He then eventually left the Congress Party and established his own party, YSR Congress Party which also matches his father's acronym,

In [8]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

vectorstore = Chroma.from_documents(
  docs, embedding_model
)

In [10]:
from langchain.chat_models import init_chat_model

In [11]:
## Here we are adding LLM and Prompt for Query Enhancement
import os
from dotenv import load_dotenv
load_dotenv


llm = init_chat_model(
    "gpt-4o-mini",
    model_provider="openai"
)



In [13]:
from langchain.vectorstores import Chroma
## creating vector store
db = Chroma.from_documents(documents = docs,embedding=embedding_model,persist_directory = "output/jagan_for_hyde.db")
##create the retriever
base_retriever=db.as_retriever(search_kwargs = {"k":5})

In [14]:

from langchain_core.output_parsers import StrOutputParser
## Generating a prompt gor generating HyDE
from langchain.prompts.chat import SystemMessagePromptTemplate, ChatPromptTemplate

def get_hyde_doc(query):
    template = """Imagine you are an expert writing a detailed explanation on the topic: '{query}'
    create a hypothetical answer for the topic"""

    system_message_prompt = SystemMessagePromptTemplate.from_template(template = template)
    chat_prompt = ChatPromptTemplate.from_messages([system_message_prompt])
    messages = chat_prompt.format_prompt(query = query).to_messages()
    print(messages)
    response = llm.invoke(messages)
    hypo_doc = response.content
    return hypo_doc

In [15]:
query= 'How many days jagan serving as cheif minister what schemems he did'
print(get_hyde_doc(query=query))

[SystemMessage(content="Imagine you are an expert writing a detailed explanation on the topic: 'How many days jagan serving as cheif minister what schemems he did'\n    create a hypothetical answer for the topic", additional_kwargs={}, response_metadata={})]
As of October 2023, Yeduguri Sandinti Jagan Mohan Reddy, commonly known as Jagan, has been serving as the Chief Minister of Andhra Pradesh since May 30, 2019. This means that as of October 2023, he has completed over 4 years (approximately 1,600 days) in office.

During his tenure, Chief Minister Jagan Mohan Reddy introduced an array of welfare schemes aimed at transforming the socio-economic landscape of Andhra Pradesh. His administration is particularly known for its ambitious and extensive welfare initiatives that target various sections of society. Here are some of the key schemes launched during his term:

1. **Amma Vodi (Mother's Lap)**: This flagship scheme provides financial assistance to poor mothers for the education of t

In [16]:
matched_doc = base_retriever.invoke(get_hyde_doc(query))
print(matched_doc)

[SystemMessage(content="Imagine you are an expert writing a detailed explanation on the topic: 'How many days jagan serving as cheif minister what schemems he did'\n    create a hypothetical answer for the topic", additional_kwargs={}, response_metadata={})]
[Document(metadata={'source': 'https://en.wikipedia.org/wiki/Y._S._Jagan_Mohan_Reddy_ministry', 'summary': 'The Y. S. Jagan Mohan Reddy ministry (or also known as 27th ministry of Andhra Pradesh) of the state of Andhra Pradesh formed the executive branch of the government of Andhra Pradesh. Along with the chief minister, there are 6 deputy chief ministers and cabinet ministers.', 'title': 'Y. S. Jagan Mohan Reddy ministry'}, page_content='The Y. S. Jagan Mohan Reddy ministry (or also known as 27th ministry of Andhra Pradesh) of the state of Andhra Pradesh formed the executive branch of the government of Andhra Pradesh. Along with the chief minister, there are 6 deputy chief ministers and cabinet ministers.\n\n\n== Cabinet Ministers

In [17]:
from langchain.chains.hyde.base import HypotheticalDocumentEmbedder

from langchain.prompts import PromptTemplate
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains.combine_documents import create_stuff_documents_chain

# Step 1: Load and split documents
loader = TextLoader("langchain_crewai_dataset.txt")
docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(docs)

In [19]:
# Step 2: Set up LLM and embeddings

from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [21]:
# Step 3: HyDE Embedder using prompt_key='web_search'
hyde_embedding_function = HypotheticalDocumentEmbedder.from_llm(
    llm=llm,
    base_embeddings=embedding_model,
    prompt_key="web_search"
)

In [22]:
# Step 4: Store documents in Chroma with HyDE embeddings
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=hyde_embedding_function,
    persist_directory="output/langchain"
)

In [23]:
# Step 5: RAG answer generation prompt
rag_prompt = PromptTemplate.from_template("""
Use the context below to answer the question.

Context:
{context}

Question: {input}
""")
rag_chain = create_stuff_documents_chain(llm=llm, prompt=rag_prompt)

In [24]:
# Step 6: Final RAG Pipeline
def hyde_rag_pipeline(query):
    matched_docs = vectorstore.similarity_search(query, k=4)
    print(matched_docs)
    response = rag_chain.invoke({
        "input": query,
        "context": matched_docs
    })
    return response

In [25]:
# Step 7: Run example query
query = "What memory modules does LangChain provide?"
answer = hyde_rag_pipeline(query)
print("✅ Final Answer:\n", answer)

[Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain offers memory modules like ConversationBufferMemory and ConversationSummaryMemory. These allow the LLM to maintain awareness of previous conversation turns or summarize long interactions to fit within token limits. (v5)'), Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain offers memory modules like ConversationBufferMemory and ConversationSummaryMemory. These allow the LLM to maintain awareness of previous conversation turns or summarize long interactions to fit within token limits. (v3)'), Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain offers memory modules like ConversationBufferMemory and ConversationSummaryMemory. These allow the LLM to maintain awareness of previous conversation turns or summarize long interactions to fit within token limits. (v2)'), Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_conte